# Topic 10 - Using JSON APIs to Automate Tasks

This notebook accompanies **Video 10** and focuses on understanding JSON and JSON APIs as a foundation for automation. We move from local JSON basics to real API calls and end with a large-language-model example via OpenRouter.


## 1. Why JSON Matters in Automation
- JSON is the dominant data exchange format for modern APIs and web services.
- It is machine readable, compact, and maps directly to Python dictionaries and lists.
- Nearly every automation task boils down to: request data (JSON) -> parse -> transform -> save or trigger actions.


## 2. What Is JSON
JSON represents data as key/value pairs (objects) and ordered collections (arrays). Common building blocks:
- Strings, numbers, booleans, and null
- Objects: `{ "name": "Ada", "city": "London" }`
- Arrays: `[1, 2, 3]` or mixed objects such as `[ {"id": 1}, {"id": 2} ]`


In [ ]:
import json

# note this is just a sample JSON string for demonstration purposes
sample_json = '''
{
  "name": "Alice",
  "age": 30,
  "skills": ["Python", "Data Analysis"],
  "active": true
}
'''

data = json.loads(sample_json)
{
    "first_skill": data["skills"][0],
    "is_active": data["active"],
    "field_count": len(data)
}


{'first_skill': 'Python', 'is_active': True, 'field_count': 4}

## 3. Reading and Writing JSON in Python
Use the `json` standard library to persist JSON to disk and bring it back. Pretty-printing (`indent=2`) keeps files readable for debugging.


In [2]:
# Writing JSON to disk
with open("sample.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

# Reading JSON back
with open("sample.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

loaded["skills"][1]


'Data Analysis'

## 4. Nested JSON Structures
Real APIs often return deeply nested JSON. Access nested dictionaries and lists carefully, using `.get()` when unsure a field exists.


In [3]:
nested = {
    "meta": {"count": 2},
    "results": [
        {"id": 1, "value": 10},
        {"id": 2, "value": 20}
    ]
}

first = nested["results"][0]
total = sum(item["value"] for item in nested["results"])
{
    "first_id": first["id"],
    "total": total,
    "count_from_meta": nested["meta"]["count"]
}


{'first_id': 1, 'total': 30, 'count_from_meta': 2}

## 5. Making HTTP Requests That Return JSON
`requests` is the go-to library for HTTP in Python. Always check status codes and call `.json()` only after a successful response. We'll use the reliable demo API at `jsonplaceholder.typicode.com`.


In [5]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts/1", timeout=10)
response.raise_for_status()  # raises if not 2xx
post = response.json()
post


{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [ ]:
{
    "title": post["title"],
    "user": post["userId"],
    "preview": post["body"][:60] + "..."
}


## 6. Query Parameters and Pagination
Pass query parameters with the `params` argument. Many APIs support pagination controls such as `_limit` and `_page` (or `page`/`per_page`).


In [7]:
params = {
    "_limit": 5,  # items per page
    "_page": 2    # which page to fetch
}

paged = requests.get("https://jsonplaceholder.typicode.com/posts", params=params, timeout=10)
paged.raise_for_status()
posts_page = paged.json()
for post in posts_page:
    print(post)
# {
#     "items_on_page": len(posts_page),
#     "first_title": posts_page[0]["title"],
#     "total_available": paged.headers.get("X-Total-Count")
# }


{'userId': 1, 'id': 6, 'title': 'dolorem eum magni eos aperiam quia', 'body': 'ut aspernatur corporis harum nihil quis provident sequi\nmollitia nobis aliquid molestiae\nperspiciatis et ea nemo ab reprehenderit accusantium quas\nvoluptate dolores velit et doloremque molestiae'}
{'userId': 1, 'id': 7, 'title': 'magnam facilis autem', 'body': 'dolore placeat quibusdam ea quo vitae\nmagni quis enim qui quis quo nemo aut saepe\nquidem repellat excepturi ut quia\nsunt ut sequi eos ea sed quas'}
{'userId': 1, 'id': 8, 'title': 'dolorem dolore est ipsam', 'body': 'dignissimos aperiam dolorem qui eum\nfacilis quibusdam animi sint suscipit qui sint possimus cum\nquaerat magni maiores excepturi\nipsam ut commodi dolor voluptatum modi aut vitae'}
{'userId': 1, 'id': 9, 'title': 'nesciunt iure omnis dolorem tempora et accusantium', 'body': 'consectetur animi nesciunt iure dolore\nenim quia ad\nveniam autem ut quam aut nobis\net est aut quod aut provident voluptas autem voluptas'}
{'userId': 1, 'id

## 7. Headers and Authentication
Many APIs require extra headers: API keys (`Authorization: Bearer ...`), content types, or custom identifiers. This example uses httpbin.org to echo back headers we send.


In [11]:
headers = {
    "Accept": "application/json",
    "X-Demo": "rtu-automation"
}

resp = requests.get(
    "https://httpbingo.org/headers",
    headers=headers,
    timeout=10
)

resp.raise_for_status()
# let's pritn all headers returned by the server
for header in resp.json()["headers"].items():
    print(header) 


('Accept', ['application/json'])
('Accept-Encoding', ['gzip, deflate'])
('Host', ['httpbingo.org'])
('User-Agent', ['python-requests/2.32.5'])
('Via', ['1.1 fly.io, 1.1 fly.io'])
('X-Demo', ['rtu-automation'])
('X-Forwarded-For', ['78.84.185.202, 66.241.125.232'])
('X-Forwarded-Port', ['443'])
('X-Forwarded-Proto', ['https'])
('X-Forwarded-Ssl', ['on'])
('X-Request-Start', ['t=1765822321755565'])


## 8. POST Requests with JSON Payloads
Send JSON payloads with the `json` parameter. The demo API accepts and echoes the payload so you can verify the request body.


In [12]:
payload = {
    "title": "foo",
    "body": "bar",
    "userId": 1
}

post_response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=payload,
    timeout=10
)
post_response.raise_for_status()
post_response.json()


{'title': 'foo', 'body': 'bar', 'userId': 1, 'id': 101}

## 9. Case Study: Weather API Pattern
Open-Meteo provides a free weather API that returns hourly time series. This mirrors real-world JSON responses: nested objects, arrays, and metadata that you must normalize before analysis.


In [16]:
# let's use Riga parameters for weather forecast
latitude = 56.9496
longitude = 24.1052
# we want temperature and humidity and wind for next day
weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m,relativehumidity_2m,windspeed_10m",
    "timezone": "Europe/Riga",
    "forecast_days": 1
}

weather_response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=weather_params,
    timeout=10
)
weather_response.raise_for_status()
weather = weather_response.json()
# let's print our forecasted temperatures and humidity for next day
for time, temp, humidity, wind in zip(
    weather["hourly"]["time"],
    weather["hourly"]["temperature_2m"],
    weather["hourly"]["relativehumidity_2m"],
    weather["hourly"]["windspeed_10m"]
):
    print(f"{time}: {temp}°C, {humidity}% RH, {wind} m/s")


2025-12-15T00:00: 4.7°C, 96% RH, 15.5 m/s
2025-12-15T01:00: 5.3°C, 96% RH, 18.0 m/s
2025-12-15T02:00: 5.8°C, 96% RH, 18.0 m/s
2025-12-15T03:00: 6.0°C, 96% RH, 18.4 m/s
2025-12-15T04:00: 6.2°C, 96% RH, 15.8 m/s
2025-12-15T05:00: 6.5°C, 96% RH, 14.4 m/s
2025-12-15T06:00: 6.5°C, 97% RH, 12.6 m/s
2025-12-15T07:00: 6.3°C, 96% RH, 13.7 m/s
2025-12-15T08:00: 6.3°C, 96% RH, 14.8 m/s
2025-12-15T09:00: 6.4°C, 96% RH, 16.2 m/s
2025-12-15T10:00: 6.4°C, 95% RH, 16.2 m/s
2025-12-15T11:00: 6.4°C, 95% RH, 14.8 m/s
2025-12-15T12:00: 6.5°C, 95% RH, 14.0 m/s
2025-12-15T13:00: 6.5°C, 94% RH, 13.0 m/s
2025-12-15T14:00: 6.6°C, 94% RH, 11.9 m/s
2025-12-15T15:00: 6.8°C, 92% RH, 12.2 m/s
2025-12-15T16:00: 6.8°C, 88% RH, 11.2 m/s
2025-12-15T17:00: 6.6°C, 89% RH, 10.1 m/s
2025-12-15T18:00: 6.4°C, 89% RH, 10.8 m/s
2025-12-15T19:00: 6.2°C, 90% RH, 11.2 m/s
2025-12-15T20:00: 6.4°C, 87% RH, 12.6 m/s
2025-12-15T21:00: 6.5°C, 85% RH, 13.0 m/s
2025-12-15T22:00: 6.1°C, 89% RH, 11.9 m/s
2025-12-15T23:00: 6.1°C, 88% RH, 1

## 10. Converting JSON to Pandas DataFrames
Normalize nested lists into tabular data for analysis. Here we convert hourly weather readings to a DataFrame and preview the first rows.


In [18]:
import pandas as pd

hourly_df = pd.DataFrame({
    "time": weather["hourly"]["time"],
    "temp_c": weather["hourly"]["temperature_2m"],
    "humidity_pct": weather["hourly"]["relativehumidity_2m"],
    "wind_m_s": weather["hourly"]["windspeed_10m"]
})

hourly_df.head()


,time,temp_c,humidity_pct,wind_m_s
0,2025-12-15T00:00,4.7,96,15.5
1,2025-12-15T01:00,5.3,96,18.0
2,2025-12-15T02:00,5.8,96,18.0
3,2025-12-15T03:00,6.0,96,18.4
4,2025-12-15T04:00,6.2,96,15.8


## 11. Saving and Reusing API Results
Cache API responses locally to avoid rate limits and speed up experiments. CSV is easy to inspect and reuse in later notebooks.


In [20]:
hourly_df.to_csv("hourly_weather.csv", index=False)

# Quick numeric summary to validate ranges
hourly_df.describe()


,temp_c,humidity_pct,wind_m_s
count,24.00000,24.000000,24.000000
mean,6.26250,92.958333,13.825000
std,0.46139,3.723943,2.401856
min,4.70000,85.000000,10.100000
25%,6.17500,89.000000,11.900000
50%,6.40000,95.000000,13.350000
75%,6.50000,96.000000,15.575000
max,6.80000,97.000000,18.400000


## 12. Rate Limits and Responsible API Use
- Read the docs for each API's rate limits and backoff recommendations.
- Cache responses during development instead of refetching on every run.
- Add timeouts to every request and retry only with delays.
- Never commit secrets (API keys) to version control; use environment variables or key vaults.


## 13. Common Failure Modes
- Invalid JSON (network hiccups or HTML error pages). Wrap `response.json()` in try/except or inspect `response.text` when parsing fails.
- Authentication errors (401/403). Double-check headers and scopes.
- Missing fields. Use `.get()` and default values when a field may be absent.
- Pagination surprises. Verify total counts and iterate until no results remain.


## 14. APIs vs Web Scraping
Prefer APIs because they are stable, documented, and faster. Scraping HTML is brittle and often violates terms of service. Use scraping only when no API exists and always respect robots.txt.


## 15. Automation Patterns Using JSON APIs
- Fetch -> clean -> analyze -> export -> notify.
- Combine multiple APIs: e.g., fetch weather, decide if a job should run, and send a Slack message.
- Validate incoming JSON schemas early to fail fast.
- Centralize common utilities (retry logic, logging, auth) to keep notebooks simple.


## 16. Calling the OpenRouter API
OpenRouter exposes an OpenAI-compatible chat endpoint for LLM-powered automation. Set `OPENROUTER_API_KEY` in your environment before running. Include helpful headers like `HTTP-Referer` and `X-Title` to identify your app.


In [21]:
import os

openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    print("Add OPENROUTER_API_KEY to your environment before running this cell.")
else:
    openrouter_headers = {
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/yourname/rtu-automation",
        "X-Title": "RTU Automation Notebook"
    }

    openrouter_payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "system", "content": "You are a concise automation assistant."},
            {"role": "user", "content": "Summarize the latest weather sample we fetched."}
        ]
    }

    openrouter_response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=openrouter_headers,
        json=openrouter_payload,
        timeout=30
    )
    openrouter_response.raise_for_status()
    openrouter_response.json()["choices"][0]["message"]["content"]


HTTPError: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions

## 17. Future Project Ideas
- Daily weather/email digest that stops a scheduled job if temperatures spike.
- Content pipeline: pull blog drafts from an API, summarize with OpenRouter, and push to a CMS endpoint.
- Support bot: read tickets from a helpdesk API, classify with an LLM, and route to the right team.
- Data quality monitor: fetch metrics from multiple APIs, merge in Pandas, alert when thresholds break.


## 18. Key Takeaways
JSON is the lingua franca of APIs. Mastering how to request, parse, normalize, and cache JSON responses unlocks reliable automation, from simple REST calls to LLM-driven workflows via OpenRouter.
